In [5]:
from nltk import CFG, ChartParser

# 定义一个简单的CFG
grammar = CFG.fromstring("""
  S -> NP VP
  NP -> Det N
  VP -> V NP
  Det -> 'the' | 'a'
  N -> 'cat' | 'dog'
  V -> 'chases' | 'sees'
""")

# 使用ChartParser解析句子
parser = ChartParser(grammar)

# 定义句子
sentence = ['the', 'cat', 'chases', 'a', 'dog']

# 解析句子并生成句法树
for tree in parser.parse(sentence):
    tree.pretty_print()

              S               
      ________|_____           
     |              VP        
     |         _____|___       
     NP       |         NP    
  ___|___     |      ___|___   
Det      N    V    Det      N 
 |       |    |     |       |  
the     cat chases  a      dog



In [27]:
from transformers import BertTokenizer, BertModel
import torch
from nltk import CFG, ChartParser

# 加载BERT模型和分词器
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# 定义句子
sentence = "The cat chased the dog".split()

# 编码句子，add_special_tokens=True 以确保添加[CLS]和[SEP]
inputs = tokenizer(sentence, return_tensors="pt", add_special_tokens=True)

# 通过BERT模型进行前向传播
with torch.no_grad():
    outputs = model(**inputs)

# 获取最后一层的词向量（[batch_size, sequence_length, hidden_size]）
word_embeddings = outputs.last_hidden_state

# 获取BERT分词后的tokens
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

# 打印分词后的tokens和对应的词向量
for i, token in enumerate(tokens):
    print(f"Token: {token}, Embedding: {word_embeddings[0, i, :5]}...")  # 打印前五个元素

# 自定义解析器类，将BERT词向量传入解析器
class ChartParserWithEmbeddings(ChartParser):
    def parse(self, tokens):
        embeddings = word_embeddings[0].cpu().numpy()  # 提取BERT的词向量并转换为numpy数组
        print("\n词向量与单词对应关系：")
        for i, word in enumerate(tokens):
            print(f"Word: {word}, Embedding: {embeddings[i][:5]}...")  # 打印前五个元素

        # 使用原始ChartParser继续解析
        return super().parse(tokens)

# 定义简单文法规则（简单句子的文法）
grammar = CFG.fromstring("""
  S -> NP VP
  NP -> Det N
  VP -> V NP
  Det -> 'the'
  N -> 'cat' | 'dog'
  V -> 'chased'
""")

# 使用自定义解析器解析句子
parser = ChartParserWithEmbeddings(grammar)

# 解析句子并生成句法树
for tree in parser.parse(sentence):
    tree.pretty_print()

Token: [CLS], Embedding: tensor([-0.2025,  0.1280, -0.0259,  0.0305,  0.1335])...
Token: the, Embedding: tensor([-0.5384, -0.7801, -0.2701, -0.2189,  0.0985])...
Token: [SEP], Embedding: tensor([ 1.0039,  0.1235, -0.3338,  0.3777, -0.3088])...

词向量与单词对应关系：
Word: The, Embedding: [-0.20248762  0.12795791 -0.02593403  0.03052175  0.13345957]...
Word: cat, Embedding: [-0.5383991  -0.78006387 -0.27009365 -0.21890375  0.09854703]...
Word: chased, Embedding: [ 1.0039455   0.12354217 -0.33383307  0.37773514 -0.30878347]...


IndexError: index 3 is out of bounds for axis 0 with size 3

In [31]:
from transformers import BertTokenizer, BertModel
import torch
from nltk import CFG, ChartParser

# 加载BERT模型和分词器
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# 定义句子
sentence = "The cat chased the dog".split()

# 编码句子
inputs = tokenizer(sentence, return_tensors="pt", add_special_tokens=True)

# 通过BERT模型进行前向传播
with torch.no_grad():
    outputs = model(**inputs)

# 获取最后一层的词向量（[batch_size, sequence_length, hidden_size]）
word_embeddings = outputs.last_hidden_state

# 获取BERT分词后的tokens
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

# 现在我们合并子词的词向量，并将它们映射回原始词
# 创建一个新的词向量列表
final_embeddings = []
final_tokens = []
current_token = ""
current_embedding = torch.zeros(word_embeddings.size(2))

# 合并子词的词向量
for i, token in enumerate(tokens):
    if token.startswith("##"):  # 如果是子词
        current_token += token[2:]  # 合并子词
        current_embedding += word_embeddings[0, i, :]
    else:
        # 如果当前token是一个完整词
        if current_token:
            final_tokens.append(current_token)
            final_embeddings.append(current_embedding)
        current_token = token  # 当前词
        current_embedding = word_embeddings[0, i, :]
        
# 添加最后一个词
if current_token:
    final_tokens.append(current_token)
    final_embeddings.append(current_embedding)

# 打印每个token及其对应的词向量
for token, embedding in zip(final_tokens, final_embeddings):
    print(f"Token: {token}, Embedding: {embedding[:5]}...")  # 打印前五个元素

# 自定义解析器类，将BERT词向量传入解析器
class ChartParserWithEmbeddings(ChartParser):
    def parse(self, tokens):
        # 使用之前合并好的词向量进行解析
        embeddings = final_embeddings  # 使用合并后的词向量
        print("\n词向量与单词对应关系：")
        for i, word in enumerate(tokens):
            print(f"Word: {word}, Embedding: {embeddings[i][:5]}...")  # 打印前五个元素

        # 使用原始ChartParser继续解析
        return super().parse(tokens)

# 定义简单文法规则（简单句子的文法）
grammar = CFG.fromstring("""
  S -> NP VP
  NP -> Det N
  VP -> V NP
  Det -> 'the'
  N -> 'cat' | 'dog'
  V -> 'chased'
""")

# 使用自定义解析器解析句子
parser = ChartParserWithEmbeddings(grammar)

# 解析句子并生成句法树
for tree in parser.parse(sentence):
    tree.pretty_print()

Token: [CLS], Embedding: tensor([-0.2025,  0.1280, -0.0259,  0.0305,  0.1335])...
Token: the, Embedding: tensor([-0.5384, -0.7801, -0.2701, -0.2189,  0.0985])...
Token: [SEP], Embedding: tensor([ 1.0039,  0.1235, -0.3338,  0.3777, -0.3088])...

词向量与单词对应关系：
Word: The, Embedding: tensor([-0.2025,  0.1280, -0.0259,  0.0305,  0.1335])...
Word: cat, Embedding: tensor([-0.5384, -0.7801, -0.2701, -0.2189,  0.0985])...
Word: chased, Embedding: tensor([ 1.0039,  0.1235, -0.3338,  0.3777, -0.3088])...


IndexError: list index out of range